# Bootstrap ppxf Summary: Combined Results

Consolidates bootstrap error estimates from notebooks 03 (z=0.67511) and
03b (z=0.67564) across all three template libraries (FSPS, EMILES, XSL).

Key analyses:
1. Compare sigma and V across templates and input redshifts
2. Investigate average residuals as a function of polynomial degree
3. Identify the optimal degree range where sigma stabilizes
4. Final recommended sigma with bootstrap error for the paper

## 1. Load all saved results

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.facecolor'] = 'white'
plt.rc('font', family='serif', size=14)
plt.rc('axes', linewidth=1.5, labelsize=16)
plt.rc('xtick', labelsize=14, direction='in')
plt.rc('ytick', labelsize=14, direction='in')

results_dir = '../results'
templates = ['fsps', 'emiles', 'xsl']
colors = {'fsps': 'C0', 'emiles': 'C1', 'xsl': 'C2'}

# Load original ppxf fits (contains best_fit, galaxy, residuals, etc.)
orig = {}
for sps in templates:
    orig[sps] = dict(np.load(f'{results_dir}/ppxf_integrated_spectrum_results_{sps}.npz',
                              allow_pickle=True))
    print(f'{sps}: {len(orig[sps]["degrees"])} degrees, sigma range: '
          f'{orig[sps]["vel_dis"].min():.0f}-{orig[sps]["vel_dis"].max():.0f} km/s')

# Load bootstrap results for both redshifts
boot = {}  # boot['fsps']['z1'] and boot['fsps']['z2']
z_labels = {'z1': 'z=0.67511', 'z2': 'z=0.67564'}
z_suffixes = {'z1': '', 'z2': '_z067564'}

for sps in templates:
    boot[sps] = {}
    for zkey, suffix in z_suffixes.items():
        path = f'{results_dir}/ppxf_bootstrap_errors_{sps}{suffix}.npz'
        try:
            boot[sps][zkey] = dict(np.load(path, allow_pickle=True))
            n = boot[sps][zkey]['n_bootstrap']
            print(f'  {sps} {z_labels[zkey]}: loaded (N={n})')
        except FileNotFoundError:
            print(f'  {sps} {z_labels[zkey]}: NOT FOUND — run notebook 03/03b first')

## 2. Residuals as a function of polynomial degree

How well does the ppxf model fit the data at each degree? The residuals
tell us whether higher degrees are actually improving the fit or just
fitting noise.

In [ ]:
# Compute residual statistics for each template and degree
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

for sps in templates:
    o = orig[sps]
    degrees = o['degrees']
    galaxy = o['galaxy']
    best_fit = o['best_fit']  # (n_deg, n_pix)
    goodpixels = o['goodpixels']
    
    # Compute per-degree residual statistics (only on good pixels)
    resid_std = np.zeros(len(degrees))
    resid_mad = np.zeros(len(degrees))  # median absolute deviation
    resid_rms = np.zeros(len(degrees))
    resid_max = np.zeros(len(degrees))
    
    for i, deg in enumerate(degrees):
        resid = galaxy - best_fit[i]
        resid_good = resid[goodpixels]
        resid_std[i] = np.std(resid_good)
        resid_mad[i] = np.median(np.abs(resid_good))
        resid_rms[i] = np.sqrt(np.mean(resid_good**2))
        resid_max[i] = np.max(np.abs(resid_good))
    
    axes[0, 0].plot(degrees, resid_std, '-o', color=colors[sps], markersize=4, label=sps)
    axes[0, 1].plot(degrees, resid_mad, '-o', color=colors[sps], markersize=4, label=sps)
    axes[1, 0].plot(degrees, resid_rms, '-o', color=colors[sps], markersize=4, label=sps)
    axes[1, 1].plot(degrees, o['fit_chi2'], '-o', color=colors[sps], markersize=4, label=sps)

axes[0, 0].set_ylabel('Residual Std Dev')
axes[0, 0].set_title('Std Dev of residuals (good pixels)')
axes[0, 1].set_ylabel('Median Abs Deviation')
axes[0, 1].set_title('MAD of residuals (good pixels)')
axes[1, 0].set_ylabel('RMS')
axes[1, 0].set_title('RMS of residuals (good pixels)')
axes[1, 1].set_ylabel(r'$\chi^2$/DOF')
axes[1, 1].set_title(r'$\chi^2$/DOF from ppxf')

for ax in axes.ravel():
    ax.set_xlabel('Additive Polynomial Degree')
    ax.legend(fontsize=10)
    ax.grid(alpha=0.3)

plt.suptitle('Residual diagnostics vs polynomial degree', fontsize=15)
plt.tight_layout()
plt.show()

In [ ]:
# Delta chi2 and delta residuals: improvement per additional degree
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for sps in templates:
    o = orig[sps]
    degrees = o['degrees']
    chi2 = o['fit_chi2']
    
    # Fractional improvement in chi2 per degree
    dchi2 = -np.diff(chi2)  # positive = improvement
    frac_dchi2 = dchi2 / chi2[:-1] * 100  # percent improvement
    
    axes[0].plot(degrees[1:], dchi2, '-o', color=colors[sps], markersize=4, label=sps)
    axes[1].plot(degrees[1:], frac_dchi2, '-o', color=colors[sps], markersize=4, label=sps)

axes[0].axhline(0, color='k', ls='--', lw=0.8)
axes[0].set_xlabel('Degree')
axes[0].set_ylabel(r'$-\Delta\chi^2$/DOF')
axes[0].set_title(r'$\chi^2$ improvement per degree (positive = better)')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].axhline(0, color='k', ls='--', lw=0.8)
axes[1].set_xlabel('Degree')
axes[1].set_ylabel(r'Fractional $\chi^2$ improvement (%)')
axes[1].set_title('Diminishing returns in fit quality')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print('Degree where chi2 improvement drops below 0.5%:')
for sps in templates:
    chi2 = orig[sps]['fit_chi2']
    dchi2_frac = -np.diff(chi2) / chi2[:-1] * 100
    stable_deg = np.where(dchi2_frac < 0.5)[0]
    if len(stable_deg) > 0:
        print(f'  {sps}: degree >= {stable_deg[0] + 1}')
    else:
        print(f'  {sps}: never stabilizes')

## 3. Sigma vs degree: all templates, both redshifts

In [ ]:
fig, ax = plt.subplots(figsize=(14, 8))

for sps in templates:
    for zkey, ls in [('z1', '-'), ('z2', '--')]:
        if zkey not in boot[sps]:
            continue
        b = boot[sps][zkey]
        degs = b['degrees']
        sigma = b['sigma_original']
        
        ax.fill_between(degs, b['sigma_p16'], b['sigma_p84'],
                        alpha=0.1, color=colors[sps])
        ax.plot(degs, sigma, ls, color=colors[sps], marker='o', markersize=4,
                label=f'{sps} ({z_labels[zkey]})')

ax.set_xlabel('Additive Polynomial Degree')
ax.set_ylabel(r'$\sigma$ (km/s)')
ax.set_title(r'Velocity Dispersion vs Degree — all templates, both input redshifts')
ax.legend(fontsize=10, ncol=2)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{results_dir}/figures/sigma_vs_degree_all.png', dpi=300, bbox_inches='tight')
plt.show()

## 4. Effect of input redshift on sigma

In [ ]:
# Delta sigma between the two input redshifts
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for sps in templates:
    if 'z1' not in boot[sps] or 'z2' not in boot[sps]:
        continue
    b1 = boot[sps]['z1']
    b2 = boot[sps]['z2']
    degs = b1['degrees']
    
    # Delta sigma
    dsigma = b2['sigma_original'] - b1['sigma_original']
    axes[0].plot(degs, dsigma, '-o', color=colors[sps], markersize=4, label=sps)
    
    # Delta V
    dV = b2['V_original'] - b1['V_original']
    axes[1].plot(degs, dV, '-o', color=colors[sps], markersize=4, label=sps)

axes[0].axhline(0, color='k', ls='--', lw=0.8)
axes[0].set_xlabel('Degree')
axes[0].set_ylabel(r'$\Delta\sigma$ (km/s)')
axes[0].set_title(r'$\sigma$(z=0.67564) $-$ $\sigma$(z=0.67511)')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].axhline(0, color='k', ls='--', lw=0.8)
axes[1].set_xlabel('Degree')
axes[1].set_ylabel(r'$\Delta V$ (km/s)')
axes[1].set_title(r'$V$(z=0.67564) $-$ $V$(z=0.67511)')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

# Summary statistics
print('Mean |delta sigma| across all degrees:')
for sps in templates:
    if 'z1' in boot[sps] and 'z2' in boot[sps]:
        ds = np.abs(boot[sps]['z2']['sigma_original'] - boot[sps]['z1']['sigma_original'])
        print(f'  {sps}: {np.mean(ds):.1f} km/s (max: {np.max(ds):.1f})')

## 5. Bootstrap error comparison across templates

In [ ]:
# Compare bootstrap errors (averaged lo/hi) across templates and redshifts
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for sps in templates:
    for zkey, ls in [('z1', '-'), ('z2', '--')]:
        if zkey not in boot[sps]:
            continue
        b = boot[sps][zkey]
        degs = b['degrees']
        sigma_err = (b['sigma_boot_err_lo'] + b['sigma_boot_err_hi']) / 2
        V_err = (b['V_boot_err_lo'] + b['V_boot_err_hi']) / 2
        
        axes[0].plot(degs, sigma_err, ls, color=colors[sps], marker='o',
                     markersize=3, label=f'{sps} ({z_labels[zkey]})' if ls=='-' else None)
        axes[1].plot(degs, V_err, ls, color=colors[sps], marker='o', markersize=3)

axes[0].set_xlabel('Degree')
axes[0].set_ylabel(r'$\delta\sigma$ (km/s)')
axes[0].set_title('Bootstrap sigma error vs degree')
axes[0].legend(fontsize=10)
axes[0].grid(alpha=0.3)

axes[1].set_xlabel('Degree')
axes[1].set_ylabel(r'$\delta V$ (km/s)')
axes[1].set_title('Bootstrap V error vs degree')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Residual wavelength structure at selected degrees

In [ ]:
# Show how residuals look at low, medium, and high polynomial degrees
deg_show = [4, 12, 20, 28]
sps_show = 'fsps'  # change to compare templates
o = orig[sps_show]
lam = o['lam_gal_rest']
galaxy = o['galaxy']
goodpix = o['goodpixels']

fig, axes = plt.subplots(len(deg_show), 1, figsize=(16, 3*len(deg_show)), sharex=True)

for ax, deg in zip(axes, deg_show):
    resid = galaxy - o['best_fit'][deg]
    ax.scatter(lam, resid, s=1, alpha=0.4, c='b', label='All pixels')
    ax.scatter(lam[goodpix], resid[goodpix], s=1, alpha=0.6, c='k', label='Good pixels')
    ax.axhline(0, color='r', ls='--', lw=0.8)
    
    # Rolling std of residuals (good pixels only)
    from scipy.ndimage import uniform_filter1d
    resid_good = np.full_like(resid, np.nan)
    resid_good[goodpix] = resid[goodpix]
    # Fill NaN for rolling window
    resid_filled = np.where(np.isfinite(resid_good), resid_good, 0)
    rolling_std = np.sqrt(uniform_filter1d(resid_filled**2, size=50, mode='reflect'))
    ax.plot(lam, rolling_std, 'orange', lw=1.5, label='Rolling RMS (50 pix)')
    ax.plot(lam, -rolling_std, 'orange', lw=1.5)
    
    std_good = np.nanstd(resid[goodpix])
    ax.set_ylabel(f'deg={deg}\nstd={std_good:.4f}')
    if ax == axes[0]:
        ax.legend(fontsize=9, ncol=3)

axes[-1].set_xlabel(r'Rest Wavelength (\AA)')
plt.suptitle(f'Residuals vs wavelength — {sps_show} templates', fontsize=14)
plt.tight_layout()
plt.show()

## 7. Recommended sigma for the paper

In [ ]:
# Final summary table: sigma at a chosen degree range, all templates, both z
# Use the "stable" degree range where chi2 improvement is marginal
deg_range = np.arange(12, 22)  # adjust based on section 2 findings

print(f"Summary for degree range {deg_range[0]}-{deg_range[-1]}")
print(f"{'Template':<8} {'z_input':>8} {'sigma_med':>9} {'err_lo':>7} {'err_hi':>7} {'V_med':>7} {'chi2_avg':>8}")
print('-' * 60)

all_sigmas = []  # collect for cross-template average

for sps in templates:
    for zkey in ['z1', 'z2']:
        if zkey not in boot[sps]:
            continue
        b = boot[sps][zkey]
        degs = b['degrees']
        mask = np.isin(degs, deg_range)
        if not np.any(mask):
            continue
        
        # Average sigma and errors over the degree range
        sigma_avg = np.mean(b['sigma_original'][mask])
        err_lo_avg = np.mean(b['sigma_boot_err_lo'][mask])
        err_hi_avg = np.mean(b['sigma_boot_err_hi'][mask])
        V_avg = np.mean(b['V_original'][mask])
        
        # Average chi2 from original fits
        o = orig[sps]
        orig_mask = np.isin(o['degrees'], deg_range)
        chi2_avg = np.mean(o['fit_chi2'][orig_mask])
        
        z_val = z_labels[zkey].split('=')[1]
        print(f"{sps:<8} {z_val:>8} {sigma_avg:9.1f} {err_lo_avg:7.1f} {err_hi_avg:7.1f} "
              f"{V_avg:7.1f} {chi2_avg:8.1f}")
        
        if zkey == 'z1':  # primary redshift
            all_sigmas.append(sigma_avg)

print(f"\nCross-template average sigma (z=0.67511): {np.mean(all_sigmas):.1f} ± {np.std(all_sigmas):.1f} km/s")
print(f"Template-to-template scatter: {np.std(all_sigmas):.1f} km/s")
print(f"This systematic is {'larger' if np.std(all_sigmas) > err_lo_avg else 'smaller'} than the bootstrap statistical error.")

In [ ]:
# Visual: horizontal bar chart of sigma estimates
fig, ax = plt.subplots(figsize=(10, 6))

y_pos = 0
y_ticks = []
y_labels = []

for sps in templates:
    for zkey in ['z1', 'z2']:
        if zkey not in boot[sps]:
            continue
        b = boot[sps][zkey]
        degs = b['degrees']
        mask = np.isin(degs, deg_range)
        if not np.any(mask):
            continue
        
        sigma_avg = np.mean(b['sigma_original'][mask])
        err_lo = np.mean(b['sigma_boot_err_lo'][mask])
        err_hi = np.mean(b['sigma_boot_err_hi'][mask])
        
        ax.errorbar(sigma_avg, y_pos, xerr=[[err_lo], [err_hi]],
                    fmt='o' if zkey=='z1' else 's',
                    color=colors[sps], markersize=10, capsize=5, lw=2)
        
        y_ticks.append(y_pos)
        y_labels.append(f'{sps} ({z_labels[zkey]})')
        y_pos += 1

ax.set_yticks(y_ticks)
ax.set_yticklabels(y_labels)
ax.set_xlabel(r'$\sigma$ (km/s)')
ax.set_title(f'Velocity dispersion estimates (averaged over degrees {deg_range[0]}-{deg_range[-1]})')
ax.grid(alpha=0.3, axis='x')
ax.axvline(np.mean(all_sigmas), color='k', ls='--', lw=1, alpha=0.5, label='Cross-template mean')
ax.legend()
plt.tight_layout()
plt.savefig(f'{results_dir}/figures/sigma_summary_bar.png', dpi=300, bbox_inches='tight')
plt.show()